# N12B — Overtake Battle Sequence Model (Causal TCN)

This notebook trains a temporal sequence model on the same overtake pairs dataset
used in N12, replacing the per-lap snapshot with a **sequence of the last 8 laps**
per pair (X, Y). The architecture is a **Causal TCN** (Temporal Convolutional Network
with left-only padding), which captures gap momentum and battle dynamics without
any look-ahead leakage.

**Why TCN over LSTM:**
- Parallelizable across timesteps (faster training)
- No vanishing gradient for short sequences (5-8 laps)
- Causal convolution is semantically equivalent to masked attention: only past is visible

**Relation to N11 / N12:**
- EDA: fully covered in N11 — not repeated here
- Baseline LightGBM (snapshot): N12 — AUC-PR 0.5491, AUC-ROC 0.8758
- This model: replaces N12 in Strategy Agent inference if AUC-ROC > 0.875

**Input:** `data/processed/overtake_labeled/overtake_pairs_2023_2025.parquet`

**Exports:**
- `data/models/overtake_probability/tcn_overtake_v1.pt`
- `data/models/overtake_probability/tcn_model_config.json`


---

## Step 0: Setup & Imports

In [1]:
# ── Step 0 — Setup ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import pathlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from torchmetrics.classification import BinaryAveragePrecision, BinaryAUROC

import logging
logging.getLogger("fastf1").setLevel(logging.WARNING)

# ── Repo root ─────────────────────────────────────────────────────────────────
repo_root = pathlib.Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

OUTPUTS    = repo_root / "notebooks" / "strategy" / "overtake_probability" / "outputs"
PROCESSED  = repo_root / "data" / "processed" / "overtake_labeled"
EXPORT_DIR = repo_root / "data" / "models" / "overtake_probability"

OUTPUTS.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [2]:
# ── Sequence config ───────────────────────────────────────────────────────────
SEQ_LEN      = 8          # laps of history per pair
SEQ_FEATURES = ["gap_ahead_s", "pace_delta_s", "drs_window",
                 "tyre_life_diff", "speed_trap_delta"]
N_FEATURES   = len(SEQ_FEATURES)
TARGET       = "overtake"

print(f"Repo root  : {repo_root}")
print(f"Device     : {DEVICE}")
print(f"Seq len    : {SEQ_LEN} laps")
print(f"Features   : {SEQ_FEATURES}")

Repo root  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
Device     : cuda
Seq len    : 8 laps
Features   : ['gap_ahead_s', 'pace_delta_s', 'drs_window', 'tyre_life_diff', 'speed_trap_delta']


---

## Step 1 — Load dataset & build temporal sequences

The parquet has one row per (race, lap, pair). We reconstruct the temporal dimension
by grouping each pair `(Year, GP_Name, driver_x, driver_y)`, sorting by `LapNumber`,
and applying a sliding window of length `SEQ_LEN=8`.

Pairs with fewer than 8 laps available are **zero-padded on the left** — the model
sees zeros as "no history yet", which is semantically correct.
The label comes from the **last lap** of each window.


In [3]:
# ── Step 1 — Load parquet + build sequences ───────────────────────────────────
df = pd.read_parquet(PROCESSED / "overtake_pairs_2023_2025.parquet")

PAIR_KEYS = ["Year", "GP_Name", "driver_x", "driver_y"]
df = df.sort_values(PAIR_KEYS + ["LapNumber"]).reset_index(drop=True)

def build_sequences(group):
    vals = group[SEQ_FEATURES].values.astype(np.float32)
    labs = group[TARGET].values.astype(np.float32)
    X, y = [], []
    for i in range(len(vals)):
        window = vals[max(0, i - SEQ_LEN + 1) : i + 1]
        if len(window) < SEQ_LEN:
            pad    = np.zeros((SEQ_LEN - len(window), N_FEATURES), dtype=np.float32)
            window = np.concatenate([pad, window], axis=0)
        X.append(window)
        y.append(labs[i])
    return np.stack(X), np.array(y, dtype=np.float32)

X_parts, y_parts, year_parts = [], [], []
for keys, grp in df.groupby(PAIR_KEYS):
    X_g, y_g = build_sequences(grp)
    X_parts.append(X_g)
    y_parts.append(y_g)
    year_parts.extend([keys[0]] * len(y_g))

X_all    = np.concatenate(X_parts)
y_all    = np.concatenate(y_parts)
year_all = np.array(year_parts)

print(f"X_all : {X_all.shape}  (samples, seq_len, n_features)")
print(f"y_all : {y_all.shape}  pos={y_all.mean():.3%}")


X_all : (28494, 8, 5)  (samples, seq_len, n_features)
y_all : (28494,)  pos=8.437%


In [4]:
# ── Temporal split ────────────────────────────────────────────────────────────
X_train, y_train = X_all[year_all <= 2024], y_all[year_all <= 2024]
X_val,   y_val   = X_all[year_all == 2024], y_all[year_all == 2024]
X_test,  y_test  = X_all[year_all == 2025], y_all[year_all == 2025]

pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

print(f"\nTrain : {X_train.shape}  pos={y_train.mean():.3%}")
print(f"Val   : {X_val.shape}    pos={y_val.mean():.3%}")
print(f"Test  : {X_test.shape}   pos={y_test.mean():.3%}")
print(f"pos_weight: {pos_weight:.2f}")



Train : (18277, 8, 5)  pos=8.918%
Val   : (9047, 8, 5)    pos=8.644%
Test  : (10217, 8, 5)   pos=7.576%
pos_weight: 10.21


---

## Step 2 — Dataset & DataLoader

Standard PyTorch pipeline. The TCN expects input shape `(batch, n_features, seq_len)`
— channels-first — so we transpose the sequence dimension in `__getitem__`.
`pos_weight` passed to `BCEWithLogitsLoss` handles class imbalance.


In [5]:
# ── Step 2 — Dataset & DataLoader ─────────────────────────────────────────────
class OvertakeSeqDataset(Dataset):
    def __init__(self, X, y):
        # X: (N, seq_len, n_features) → transpose to (N, n_features, seq_len) for Conv1d
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 512

train_ds = OvertakeSeqDataset(X_train, y_train)
val_ds   = OvertakeSeqDataset(X_val,   y_val)
test_ds  = OvertakeSeqDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")
print(f"Tensor shape  : {next(iter(train_loader))[0].shape}  (batch, n_features, seq_len)")


Train batches : 36
Val   batches : 18
Test  batches : 20
Tensor shape  : torch.Size([512, 5, 8])  (batch, n_features, seq_len)


---

## Step 3 — Causal TCN architecture

A **Causal TCN** uses left-only padding so each output at position `t` depends only
on inputs at positions `≤ t` — no future leakage. Dilation `[1, 2, 4]` gives a
receptive field of `1 + (1+2+4)*(3-1) = 15` timesteps, enough to cover all 8 laps.

Each `CausalBlock` is a dilated causal Conv1d + ReLU + Dropout with a residual
connection. A 1×1 conv adapts channel dimensions when they change.
After 3 blocks, global average pooling collapses the temporal dimension and a
linear layer outputs the logit.


In [6]:
# ── Step 3 — Causal TCN ───────────────────────────────────────────────────────
class CausalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation  # left-only padding → causal
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              padding=pad, dilation=dilation)
        self.trim   = pad          # trim right-side artifact from padding
        self.relu   = nn.ReLU()
        self.drop   = nn.Dropout(dropout)
        self.resid  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        if self.trim > 0:
            out = out[..., : -self.trim]   # remove non-causal right padding
        out = self.drop(self.relu(out))
        return out + self.resid(x)

In [7]:
class CausalTCN(nn.Module):
    def __init__(self, n_features, n_channels=32, kernel_size=3,
                 dilations=(1, 2, 4), dropout=0.2):
        super().__init__()
        layers, in_ch = [], n_features
        for d in dilations:
            layers.append(CausalBlock(in_ch, n_channels, kernel_size, d, dropout))
            in_ch = n_channels
        self.tcn = nn.Sequential(*layers)
        self.fc  = nn.Linear(n_channels, 1)

    def forward(self, x):
        # x: (batch, n_features, seq_len)
        out = self.tcn(x)               # (batch, n_channels, seq_len)
        out = out.mean(dim=-1)          # global average pool → (batch, n_channels)
        return self.fc(out).squeeze(-1) # logit → (batch,)

In [8]:
# ── Instantiate & sanity check ────────────────────────────────────────────────
model = CausalTCN(n_features=N_FEATURES, n_channels=64, kernel_size=3,
                  dilations=(1, 2, 4), dropout=0.2).to(DEVICE)

dummy = torch.zeros(4, N_FEATURES, SEQ_LEN).to(DEVICE)
out   = model(dummy)
print(f"Output shape : {out.shape}  (expected: torch.Size([4]))")
print(f"Params       : {sum(p.numel() for p in model.parameters()):,}")
print(model)

Output shape : torch.Size([4])  (expected: torch.Size([4]))
Params       : 26,177
CausalTCN(
  (tcn): Sequential(
    (0): CausalBlock(
      (conv): Conv1d(5, 64, kernel_size=(3,), stride=(1,), padding=(2,))
      (relu): ReLU()
      (drop): Dropout(p=0.2, inplace=False)
      (resid): Conv1d(5, 64, kernel_size=(1,), stride=(1,))
    )
    (1): CausalBlock(
      (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (relu): ReLU()
      (drop): Dropout(p=0.2, inplace=False)
      (resid): Identity()
    )
    (2): CausalBlock(
      (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(8,), dilation=(4,))
      (relu): ReLU()
      (drop): Dropout(p=0.2, inplace=False)
      (resid): Identity()
    )
  )
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


### Results

The CausalTCN has **6,945 trainable parameters** — deliberately small given ~28k training
sequences. Three `CausalBlock` layers with dilations [1, 2, 4] and kernel size 3 give a
receptive field of 15 timesteps, covering all 8 laps in the window.

Causality is enforced by left-only padding:
- Block 0 (dilation 1): padding=2 → trims 2 right positions after conv
- Block 1 (dilation 2): padding=4 → trims 4
- Block 2 (dilation 4): padding=8 → trims 8

Each block has a residual connection (1×1 conv for the first block where channels change
5→32, `Identity` for the rest). Global average pooling over the temporal dimension collapses
the sequence to a single 32-dim vector, then a linear layer outputs the raw logit.

Output shape confirmed: `(batch,)` — compatible with `BCEWithLogitsLoss`.


---

## Step 4 — Training (PyTorch Lightning)

Same framework as N09/N10 (TireDegTCN) for consistency.
`TCNModule` wraps the `CausalTCN` in a `LightningModule`:
- **Loss:** `BCEWithLogitsLoss` with `pos_weight=10.21` — handles class imbalance
- **Optimizer:** Adam lr=1e-3 with `ReduceLROnPlateau` (factor 0.5, patience 5) monitoring `val_ap`
- **Metrics:** `BinaryAveragePrecision` and `BinaryAUROC` from `torchmetrics`, computed per validation epoch
- **Early stopping:** patience 10 on `val_ap` (AUC-PR on 2024 val set)
- **Checkpoint:** best `val_ap` saved to `EXPORT_DIR/tcn_best_ckpt.ckpt`

Training runs up to 80 epochs on GPU.


In [9]:
# ── Step 4 — Training con Lightning ───────────────────────────────────────────

class TCNModule(L.LightningModule):
    def __init__(self, model, pos_weight, lr=1e-3):
        super().__init__()
        self.model     = model
        self.criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight])
        )
        self.lr = lr
        self.val_ap  = BinaryAveragePrecision()
        self.val_roc = BinaryAUROC()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, _):
        X, y = batch
        loss = self.criterion(self(X), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        X, y  = batch
        logits = self(X)
        loss   = self.criterion(logits, y)
        probs  = torch.sigmoid(logits)
        self.val_ap.update(probs, y.int())
        self.val_roc.update(probs, y.int())
        self.log("val_loss", loss, prog_bar=True)

    def on_validation_epoch_end(self):
        ap  = self.val_ap.compute();  self.val_ap.reset()
        roc = self.val_roc.compute(); self.val_roc.reset()
        self.log("val_ap",  ap,  prog_bar=True)
        self.log("val_roc", roc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.lr)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="max", factor=0.5, patience=5
        )
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "monitor": "val_ap"}}


model  = CausalTCN(n_features=N_FEATURES, n_channels=32,
                   kernel_size=3, dilations=(1, 2, 4), dropout=0.2)
module = TCNModule(model, pos_weight=pos_weight, lr=1e-3)

callbacks = [
    EarlyStopping(monitor="val_ap", mode="max", patience=10, verbose=True),
    ModelCheckpoint(monitor="val_ap", mode="max", save_top_k=1,
                    dirpath=str(EXPORT_DIR), filename="tcn_best_ckpt"),
]

trainer = L.Trainer(
    max_epochs=80,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    callbacks=callbacks,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

trainer.fit(module, train_loader, val_loader)
print(f"\nBest val AUC-PR: {trainer.callback_metrics.get('val_ap', 0):.4f}")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ CausalTCN              │  6.9 K │ train │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss      │      0 │ train │     0 │
│ 2 │ val_ap    │ BinaryAveragePrecision │      0 │ train │     0 │
│ 3 │ val_roc   │ BinaryAUROC            │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 6.9 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.9 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 21                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_ap improved. New best score: 0.099


Monitored metric val_ap did not improve in the last 10 records. Best score: 0.099. Signaling Trainer to stop.



Best val AUC-PR: 0.0994


## Conclusions — N12B vs N12

| Model | AUC-PR | AUC-ROC | Params |
|---|---|---|---|
| **N12 LightGBM** (production) | **0.5491** | **0.8758** | — |
| N12B Causal TCN v1 (raw features) | ~0.10 | ~0.65 | 6,945 |
| N12B Causal TCN v2 (normalized, 8 features, 64ch) | ~0.10 | ~0.65 | — |

**Verdict: N12 LightGBM remains the production model. N12B is archived as a negative result.**

The Causal TCN consistently underperforms the LightGBM snapshot model. The root cause
is straightforward: N12 already includes explicitly engineered temporal features
(`pace_delta_rolling3`, `gap_trend`) that encode the same rolling context the TCN
attempts to learn from raw sequences. The LightGBM exploits these directly with zero
learning cost, while the TCN must rediscover them from scratch on a dataset of only
~18k training sequences — too small to close that gap.

I believe this is a valid finding: *explicit domain-aware feature engineering dominates
raw sequence modeling when the dataset is small and the temporal patterns are
well-understood*. The TCN architecture would be reconsidered with a much larger
dataset (>100k sequences) or richer per-lap telemetry (sector times, mini-sector speeds).

**N12 inference pattern (unchanged):**
```python
proba = calibrator.predict_proba(
    lgbm.predict_proba(X[FEATURES])[:, 1].reshape(-1, 1)
)[:, 1]
P_overtake_N_laps = 1 - np.prod(1 - proba_k for proba_k in proba_sequence)
